#  Twitter Sentiment Model Training
**Pipeline:** Tokenizer → HashingTF → IDF → Logistic Regression


## 1. Imports

In [ ]:
import os
import sys
from pathlib import Path

# Set Hadoop env before any PySpark import (same fix as consumer.py and views.py)
os.environ.setdefault('HADOOP_HOME',           r'C:\hadoop')
os.environ.setdefault('PYSPARK_PYTHON',        sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)
_hadoop_bin = r'C:\hadoop\bin'
if _hadoop_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = _hadoop_bin + os.pathsep + os.environ.get('PATH', '')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lower, trim
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, IDF, StringIndexer, IndexToString
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
import os
print("HADOOP_HOME       :", os.environ.get("HADOOP_HOME"))
print("winutils.exe found:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll found  :", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

HADOOP_HOME       : C:\hadoop
winutils.exe found: True
hadoop.dll found  : True


## 2. Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("TwitterSentimentTraining")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir",     "C:/tmp/spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

## 3. Load & Inspect Data

In [4]:
# Resolve absolute paths to the CSVs (notebook lives in ML-PySpark-Model/notebooks/)
DATASETS_DIR = (Path.cwd().parent / "datasets").resolve()

# PySpark on Windows requires forward slashes
TRAIN_PATH = str(DATASETS_DIR / "twitter_training.csv").replace("\\", "/")
VAL_PATH   = str(DATASETS_DIR / "twitter_validation.csv").replace("\\", "/")

# Fail fast with a clear error if the files aren't where we expect
assert os.path.exists(TRAIN_PATH), f"Training CSV not found at: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH),   f"Validation CSV not found at: {VAL_PATH}"

print("Train CSV:", TRAIN_PATH)
print("Val CSV  :", VAL_PATH)

COLS = ["tweet_id", "entity", "sentiment", "text"]

train_df = spark.read.csv(TRAIN_PATH, header=False, inferSchema=True).toDF(*COLS)
val_df   = spark.read.csv(VAL_PATH,   header=False, inferSchema=True).toDF(*COLS)

# Drop rows with missing text or label
train_df = train_df.dropna(subset=["text", "sentiment"])
val_df   = val_df.dropna(subset=["text", "sentiment"])

print(f"\nTrain rows : {train_df.count()}")
print(f"Val rows   : {val_df.count()}")
print()
train_df.groupBy("sentiment").count().orderBy("count", ascending=False).show()

Train CSV: D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/datasets/twitter_training.csv
Val CSV  : D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/datasets/twitter_validation.csv

Train rows : 73996
Val rows   : 1000

+----------+-----+
| sentiment|count|
+----------+-----+
|  Negative|22358|
|  Positive|20655|
|   Neutral|18108|
|Irrelevant|12875|
+----------+-----+



## 4. Text Cleaning
Strips URLs, @mentions, and non-alphabetic characters. Lowercases everything.

In [5]:
def clean_text(df):
    return (
        df
        .withColumn("text", regexp_replace(col("text"), r"http\S+", ""))
        .withColumn("text", regexp_replace(col("text"), r"@\w+", ""))
        .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z\s]", ""))
        .withColumn("text", trim(lower(col("text"))))
    )

train_df = clean_text(train_df)
val_df   = clean_text(val_df)

train_df.select("sentiment", "text").show(5, truncate=90)

+---------+-------------------------------------------------------+
|sentiment|                                                   text|
+---------+-------------------------------------------------------+
| Positive|    im getting on borderlands and i will murder you all|
| Positive|     i am coming to the borders and i will kill you all|
| Positive|      im getting on borderlands and i will kill you all|
| Positive|     im coming on borderlands and i will murder you all|
| Positive|im getting on borderlands  and i will murder you me all|
+---------+-------------------------------------------------------+
only showing top 5 rows



## 5. Build ML Pipeline
Each stage transforms the data and passes it to the next:
- **StringIndexer** — converts sentiment string to numeric label
- **Tokenizer** — splits text into word list
- **NGram (n=2)** — creates bigrams (e.g. "absolutely loved") alongside unigrams
- **HashingTF** — maps word/bigram list to fixed-size feature vector
- **IDF** — down-weights words that appear in many tweets (like 'the', 'a')
- **LogisticRegression** — multi-class classifier on the TF-IDF features

In [ ]:
label_indexer = StringIndexer(
    inputCol="sentiment", outputCol="label", handleInvalid="skip"
)

tokenizer = Tokenizer(inputCol="text", outputCol="words")

ngram = NGram(n=2, inputCol="words", outputCol="bigrams")

hashing_tf = HashingTF(
    inputCol="bigrams", outputCol="raw_features", numFeatures=20000
)

idf = IDF(
    inputCol="raw_features", outputCol="features", minDocFreq=5
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    regParam=0.01
)

pipeline = Pipeline(stages=[label_indexer, tokenizer, ngram, hashing_tf, idf, lr])
print("Pipeline stages:", [s.__class__.__name__ for s in pipeline.getStages()])

## 6. Train
Fits all pipeline stages on the training data. Takes 2–4 minutes.

In [7]:
print("Training started...")
model = pipeline.fit(train_df)
print("Training complete.")

# Label mapping produced by StringIndexer
labels = model.stages[0].labels
print("Label mapping:", {i: lbl for i, lbl in enumerate(labels)})

Training started...
Training complete.
Label mapping: {0: 'Negative', 1: 'Positive', 2: 'Neutral', 3: 'Irrelevant'}


## 7. Evaluate on Validation Set

In [8]:
predictions = model.transform(val_df)

# Map numeric predictions back to label names for readability
index_to_label = IndexToString(
    inputCol="prediction", outputCol="predicted_label",
    labels=model.stages[0].labels
)
predictions = index_to_label.transform(predictions)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

print("\nConfusion Matrix (actual vs predicted):")
predictions.groupBy("sentiment", "predicted_label") \
           .count() \
           .orderBy("sentiment", "predicted_label") \
           .show(20)

Validation Accuracy: 82.20%

Confusion Matrix (actual vs predicted):
+----------+---------------+-----+
| sentiment|predicted_label|count|
+----------+---------------+-----+
|Irrelevant|     Irrelevant|  131|
|Irrelevant|       Negative|   14|
|Irrelevant|        Neutral|    5|
|Irrelevant|       Positive|   22|
|  Negative|     Irrelevant|    5|
|  Negative|       Negative|  236|
|  Negative|        Neutral|    6|
|  Negative|       Positive|   19|
|   Neutral|     Irrelevant|    9|
|   Neutral|       Negative|   27|
|   Neutral|        Neutral|  217|
|   Neutral|       Positive|   32|
|  Positive|     Irrelevant|    8|
|  Positive|       Negative|   13|
|  Positive|        Neutral|   18|
|  Positive|       Positive|  238|
+----------+---------------+-----+



## 8. Save Model
Saves the entire fitted pipeline (not just the LR weights) so the consumer can apply identical preprocessing at inference time.

In [9]:
# Use absolute path here too, for the same reason as cell 6
MODEL_PATH = str((Path.cwd().parent / "saved_models" / "spark_lr_pipeline").resolve()).replace("\\", "/")

model.write().overwrite().save(MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

Model saved → D:/Twitter_Sentiment_analysis/Real-Time-Twitter-Sentiment-Analysis/ML-PySpark-Model/saved_models/spark_lr_pipeline


## 9. Model Comparison — Spark MLlib vs VADER

Accuracy alone is misleading on this dataset: the four classes are imbalanced, and one of them
(**Irrelevant**, 17.2% of the validation set) is a class VADER can never predict. So this section
reports **precision, recall and F1** per class, plus macro and weighted averages, for both models.

Two methodological points that make this a fair comparison:

1. **VADER reads the raw text, Spark reads the cleaned text.** `clean_text()` strips punctuation,
   emoji and capitalisation — exactly the signals VADER uses to judge intensity (`!!!`, ALL CAPS,
   emoticons). Scoring VADER on cleaned text would understate it. This mirrors what the live
   pipeline does: `consumer.py` runs VADER on `original_text` and Spark on the cleaned column.
2. **Both a 4-class and a 3-class comparison are reported.** The 4-class result is the real task
   and shows why a trained model was needed. The 3-class result (Irrelevant rows removed) is the
   fair head-to-head on the task VADER was actually designed for.

> **Note on sample size:** the validation set is 1,000 rows, so an accuracy near 86% carries roughly
> ±2.1 percentage points at 95% confidence. Small differences should not be over-interpreted.

In [ ]:
### 9.1  Score the validation set with both models
# val_df was cleaned in place back in section 4, so the original text is gone.
# Reload the CSV and keep BOTH versions side by side: raw_text for VADER, text for Spark.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

val_eval = spark.read.csv(VAL_PATH, header=False, inferSchema=True).toDF(*COLS)
val_eval = val_eval.dropna(subset=["text", "sentiment"])
val_eval = val_eval.withColumn("raw_text", col("text"))   # preserve before cleaning
val_eval = clean_text(val_eval)                            # cleans the 'text' column only

# Spark predictions (the pipeline's StringIndexer needs the 'sentiment' column, which we have)
eval_preds = model.transform(val_eval)
eval_preds = IndexToString(
    inputCol="prediction", outputCol="predicted_label",
    labels=model.stages[0].labels
).transform(eval_preds)

# 1,000 rows — small enough to pull to the driver for VADER scoring
rows = eval_preds.select("sentiment", "predicted_label", "raw_text").collect()

_analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    """Same thresholds as consumer.py, so offline numbers match live behaviour."""
    if text is None or not text.strip():
        return "Neutral"
    compound = _analyzer.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return "Positive"
    if compound <= -0.05:
        return "Negative"
    return "Neutral"

# (ground_truth, spark_prediction, vader_prediction) for every validation row
results = [(r["sentiment"], r["predicted_label"], vader_label(r["raw_text"])) for r in rows]
print(f"Scored {len(results)} validation tweets with both models.")

In [ ]:
### 9.2  Metric helpers
# Computed from first principles rather than sklearn — keeps the environment dependency-free
# and makes the definitions explicit:
#   precision = TP / (TP + FP)   "of what I labelled X, how much really was X"
#   recall    = TP / (TP + FN)   "of all the real X, how much did I find"
#   F1        = harmonic mean of the two
# macro avg  = unweighted mean over classes  -> every class counts equally (honest under imbalance)
# weighted avg = mean weighted by support    -> reflects real-world class frequencies

def compute_metrics(pairs):
    """pairs: list of (y_true, y_pred) label strings."""
    labels = sorted({t for t, _ in pairs} | {p for _, p in pairs})
    n = len(pairs)
    per_class = {}
    for lb in labels:
        tp = sum(1 for t, p in pairs if t == lb and p == lb)
        fp = sum(1 for t, p in pairs if t != lb and p == lb)
        fn = sum(1 for t, p in pairs if t == lb and p != lb)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall    = tp / (tp + fn) if tp + fn else 0.0
        f1        = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[lb] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}

    # Averages are taken over classes that actually occur in the ground truth
    present = [v for v in per_class.values() if v["support"] > 0]
    total_support = sum(v["support"] for v in present)
    macro = {k: sum(v[k] for v in present) / len(present)
             for k in ("precision", "recall", "f1")}
    weighted = {k: sum(v[k] * v["support"] for v in present) / total_support
                for k in ("precision", "recall", "f1")}

    return {
        "accuracy": sum(1 for t, p in pairs if t == p) / n,
        "per_class": per_class,
        "macro": macro,
        "weighted": weighted,
        "n": n,
    }


def print_report(title, m):
    print(f"\n{title}  (n={m['n']})")
    print("-" * 52)
    print(f"Accuracy: {m['accuracy'] * 100:.2f}%\n")
    print(f"{'class':<14}{'precision':>10}{'recall':>9}{'f1':>8}{'support':>9}")
    off_scope = 0
    for lb, v in sorted(m["per_class"].items()):
        if v["support"] == 0:
            # Model predicted a class that does not exist in this evaluation slice.
            # Those predictions are still counted as errors in accuracy and in other
            # classes' recall; listing a zero row here would just be noise.
            off_scope += 1
            continue
        print(f"{lb:<14}{v['precision']:>10.3f}{v['recall']:>9.3f}{v['f1']:>8.3f}{v['support']:>9}")
    print("-" * 52)
    print(f"{'macro avg':<14}{m['macro']['precision']:>10.3f}{m['macro']['recall']:>9.3f}{m['macro']['f1']:>8.3f}")
    print(f"{'weighted avg':<14}{m['weighted']['precision']:>10.3f}{m['weighted']['recall']:>9.3f}{m['weighted']['f1']:>8.3f}")
    if off_scope:
        print(f"\n(+{off_scope} class(es) predicted but absent from this slice's ground truth; "
              f"counted as errors.)")


def print_confusion(title, pairs):
    labels = sorted({t for t, _ in pairs} | {p for _, p in pairs})
    print(f"\n{title} — confusion matrix (rows = actual, cols = predicted)")
    print(f"{'':<13}" + "".join(f"{lb[:9]:>11}" for lb in labels))
    for actual in labels:
        if not any(t == actual for t, _ in pairs):
            continue
        counts = [sum(1 for t, p in pairs if t == actual and p == pred) for pred in labels]
        print(f"{actual:<13}" + "".join(f"{c:>11}" for c in counts))

In [ ]:
### 9.3  Four-class comparison — the full task
print("=" * 52)
print("4-CLASS: Positive / Negative / Neutral / Irrelevant")
print("=" * 52)

spark_4 = compute_metrics([(truth, sp) for truth, sp, vd in results])
vader_4 = compute_metrics([(truth, vd) for truth, sp, vd in results])

print_report("Spark MLlib (LR + TF-IDF bigrams)", spark_4)
print_report("VADER (rule-based lexicon)", vader_4)

print_confusion("\nSpark MLlib", [(truth, sp) for truth, sp, vd in results])
print_confusion("VADER",        [(truth, vd) for truth, sp, vd in results])

In [ ]:
### 9.4  Three-class comparison — the fair head-to-head
# VADER has no concept of "Irrelevant", so the 4-class numbers above penalise it for a
# capability it never claimed. Removing those rows gives the comparison VADER deserves.
# Spark's own predictions are left untouched: if it still says "Irrelevant" on one of these
# rows, that counts as an error (see the footnote in the report).
print("=" * 52)
print("3-CLASS: Irrelevant rows excluded — VADER's home turf")
print("=" * 52)

subset = [(truth, sp, vd) for truth, sp, vd in results if truth != "Irrelevant"]
print(f"Rows retained: {len(subset)} of {len(results)} "
      f"({len(results) - len(subset)} Irrelevant rows removed)\n")

spark_3 = compute_metrics([(truth, sp) for truth, sp, vd in subset])
vader_3 = compute_metrics([(truth, vd) for truth, sp, vd in subset])

print_report("Spark MLlib (LR + TF-IDF bigrams)", spark_3)
print_report("VADER (rule-based lexicon)", vader_3)

In [ ]:
### 9.5  Where the two models disagree
# The dashboard flags agreement/disagreement on every row, so it is worth quantifying:
# when the two models conflict, which one should you trust?
from collections import Counter

agree    = [(t, s, v) for t, s, v in results if s == v]
disagree = [(t, s, v) for t, s, v in results if s != v]

print(f"Agreement rate: {len(agree) / len(results) * 100:.1f}%  "
      f"({len(agree)} of {len(results)} tweets)\n")
print(f"When they agree ({len(agree)} tweets)")
print(f"  Both correct        : {sum(1 for t, s, v in agree if t == s) / len(agree) * 100:.1f}%")
print(f"\nWhen they disagree ({len(disagree)} tweets)")
print(f"  Spark correct       : {sum(1 for t, s, v in disagree if t == s) / len(disagree) * 100:.1f}%")
print(f"  VADER correct       : {sum(1 for t, s, v in disagree if t == v) / len(disagree) * 100:.1f}%")
print(f"  Both wrong          : {sum(1 for t, s, v in disagree if t != s and t != v) / len(disagree) * 100:.1f}%")

# The Irrelevant blind spot, made concrete
print("\nWhat VADER predicts on the 172 genuinely 'Irrelevant' tweets:")
for label, count in Counter(v for t, s, v in results if t == "Irrelevant").most_common():
    print(f"  {label:<12}{count:>5}")
print("  -> VADER cannot output 'Irrelevant', so every one of these is necessarily wrong.")

### 9.6  Reading the results

**Headline: Spark MLlib wins decisively, and it wins on the fair comparison too.**

| Metric (validation set) | Spark MLlib | VADER |
|---|---|---|
| Accuracy — 4-class, n=1000 | **86.10%** | 40.00% |
| Macro F1 — 4-class | **0.861** | 0.307 |
| Weighted F1 — 4-class | **0.861** | 0.337 |
| Accuracy — 3-class, n=828 | **86.47%** | 48.31% |
| Macro F1 — 3-class | **0.876** | 0.454 |

Removing the Irrelevant class lifts VADER from 40.0% to 48.3% — a real gain, and confirmation
that part of its 4-class score was an artefact of being asked a question it cannot answer. But it
is still ~38 points behind Spark on its own home turf, so the ranking is not an artefact of an
unfair setup.

**Where each model actually fails**

- **VADER's Neutral recall is 0.175.** It finds barely one in six neutral tweets, scattering the
  rest across Positive and Negative. A lexicon scorer only reports Neutral when a tweet contains
  almost no charged vocabulary, but human annotators call a tweet neutral based on *intent* —
  factual statements, questions, and announcements often contain charged words while carrying no
  opinion. This mismatch, not vocabulary coverage, is VADER's core weakness here.
- **VADER's Irrelevant recall is 0 by construction** — of the 172 Irrelevant tweets it labelled
  87 Positive, 67 Negative and 18 Neutral. Relevance is a topical judgement, not a polarity one,
  so no lexicon can recover it.
- **Spark's weakest class is Positive precision (0.791).** It over-predicts Positive: 29 Neutral
  and 20 Negative tweets get pulled into it. Since Positive recall is high (0.903), the model
  leans toward Positive when uncertain. Raising the decision threshold for that class, or adding
  class weights, would be the natural next tuning step.

**Model agreement — the interesting one for the dashboard**

The two models agree on only **38.0%** of tweets, and when they agree they are right **90.3%** of
the time — so agreement is a genuinely useful confidence signal. When they disagree, **Spark is
correct 83.5%** of the time versus VADER's **9.2%**. That justifies the pipeline's design choice of
treating Spark as the primary label and VADER as a secondary indicator rather than an equal vote.

**Caveat.** All figures come from a 1,000-row validation set (828 for the 3-class slice), so
differences of 1–2 percentage points are within noise. The gaps reported above are far larger than
that margin, so the conclusion is safe — but avoid quoting these numbers to two decimal places as
though they were precise.